In [1]:
print('ok')

ok


In [2]:
# %%
"""
============================================================================
ADVANCED RAG SYSTEM — Production Grade (Fixed)
============================================================================
Modes: Naive RAG | Advanced RAG | Corrective RAG (CRAG) | Self-RAG

Features:
  - Query Rewriting (multi-query expansion)
  - Hybrid Search (BM25 + Vector with weighted RRF fusion)
  - Cross-Encoder Reranking
  - Metadata Filtering
  - Document-aware Chunk Optimization
  - Context Compression (parallel, with citation preservation)
  - Source Attribution (file, page, chunk_id)
  - Persistent Hybrid Index (BM25 + vector kept in sync)
  - Deduplication (same file won't be ingested twice)
  - Wikipedia fallback with truncation

============================================================================
"""

# %%
from __future__ import annotations

import hashlib
import logging
import os
import re
import threading
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple, TypedDict

# %%
from dotenv import load_dotenv

load_dotenv()

# %%
# ----------------------------------------------------------------------------
# Logging
# ----------------------------------------------------------------------------
logging.basicConfig(
    level=os.getenv("RAG_LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
)
logger = logging.getLogger("rag")

# %%
# ----------------------------------------------------------------------------
# LangChain / LangGraph imports
# ----------------------------------------------------------------------------
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.document_loaders import (
    Docx2txtLoader,
    PyMuPDFLoader,
    TextLoader,
    WebBaseLoader,
)
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import END, StateGraph

from langchain_groq import ChatGroq

# Use updated packages if available, fall back to community
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma

# %%
# Optional dependencies (graceful degradation)
try:
    from rank_bm25 import BM25Okapi
    BM25_AVAILABLE = True
except ImportError:
    BM25_AVAILABLE = False
    logger.warning("rank_bm25 not installed — hybrid search will fall back to vector-only.")

try:
    from sentence_transformers import CrossEncoder
    CROSS_ENCODER_AVAILABLE = True
except ImportError:
    CROSS_ENCODER_AVAILABLE = False
    logger.warning("sentence-transformers not installed — reranking disabled.")

# %%
# ============================================================================
# CONFIGURATION
# ============================================================================
try:
    _THIS_FILE = Path(__file__).resolve()
    _NOTEBOOK_DIR = _THIS_FILE.parent
except NameError:
    _NOTEBOOK_DIR = Path.cwd()

_PROJECT_ROOT = _NOTEBOOK_DIR.parent


@dataclass(frozen=True)
class RAGConfig:
    # Paths
    data_dir: Path = field(default_factory=lambda: _PROJECT_ROOT / "data")
    vector_store_dir: Path = field(default_factory=lambda: _PROJECT_ROOT / "data" / "vector_store")

    # Supported file extensions for auto-ingestion
    supported_extensions: Tuple[str, ...] = (".pdf", ".docx", ".txt")

    # Embedding & LLM
    embedding_model: str = "all-MiniLM-L6-v2"
    llm_model: str = "openai/gpt-oss-120b"
    llm_temperature: float = 0.1
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    # Chunking
    chunk_size: int = 1000
    chunk_overlap: int = 200
    pdf_chunk_size: int = 1200
    pdf_chunk_overlap: int = 250
    web_chunk_size: int = 800
    web_chunk_overlap: int = 150

    # Retrieval
    initial_k: int = 20
    rerank_top_k: int = 5
    bm25_weight: float = 0.5
    rrf_k: int = 60

    # Context Compression
    compression_max_workers: int = 4
    compression_enabled: bool = True
    compress_request_delay_s: float = 0.1

    # CRAG
    crag_high_confidence: float = 0.6
    crag_low_confidence: float = 0.3
    crag_max_retries: int = 2

    # Wikipedia response length cap (chars)
    wikipedia_max_chars: int = 3000


CONFIG = RAGConfig()

CONFIG.data_dir.mkdir(parents=True, exist_ok=True)
CONFIG.vector_store_dir.mkdir(parents=True, exist_ok=True)

logger.info(f"Data directory   : {CONFIG.data_dir}")
logger.info(f"Vector store dir : {CONFIG.vector_store_dir}")

# %%
# ============================================================================
# CORE COMPONENTS (singletons)
# ============================================================================
embedding_model = HuggingFaceEmbeddings(model_name=CONFIG.embedding_model)

llm = ChatGroq(
    temperature=CONFIG.llm_temperature,
    model_name=CONFIG.llm_model,
)

wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=1,
        doc_content_chars_max=CONFIG.wikipedia_max_chars,
    )
)

vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory=str(CONFIG.vector_store_dir),
    collection_name="rag_documents",
    collection_metadata={"hnsw:space": "cosine"},
)

reranker: Optional[CrossEncoder] = None
if CROSS_ENCODER_AVAILABLE:
    try:
        reranker = CrossEncoder(CONFIG.reranker_model)
        logger.info(f"Cross-encoder reranker loaded: {CONFIG.reranker_model}")
    except Exception as e:
        logger.warning(f"Could not load cross-encoder: {e}")

# %%
# ============================================================================
# CHUNK OPTIMIZATION
# ============================================================================
class ChunkOptimizer:
    """Selects the appropriate splitter per document type."""

    @staticmethod
    def get_splitter(file_type: str) -> RecursiveCharacterTextSplitter:
        if file_type == "pdf":
            return RecursiveCharacterTextSplitter(
                chunk_size=CONFIG.pdf_chunk_size,
                chunk_overlap=CONFIG.pdf_chunk_overlap,
                separators=["\n\n", "\n", ". ", "? ", "! ", "; ", ", ", " ", ""],
                length_function=len,
            )
        if file_type == "url":
            return RecursiveCharacterTextSplitter(
                chunk_size=CONFIG.web_chunk_size,
                chunk_overlap=CONFIG.web_chunk_overlap,
                separators=["\n\n", "\n", ". ", " ", ""],
                length_function=len,
            )
        return RecursiveCharacterTextSplitter(
            chunk_size=CONFIG.chunk_size,
            chunk_overlap=CONFIG.chunk_overlap,
            separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
            length_function=len,
        )

    @staticmethod
    def split(documents: List[Document], file_type: str) -> List[Document]:
        splitter = ChunkOptimizer.get_splitter(file_type)
        chunks = splitter.split_documents(documents)
        return [c for c in chunks if len(c.page_content.strip()) >= 50]


# %%
# ============================================================================
# DOCUMENT PROCESSOR
# ============================================================================
class DocumentProcessor:
    @staticmethod
    def _enrich_metadata(
        docs: List[Document],
        source_id: str,
        source_name: str,
        file_type: str,
    ) -> List[Document]:
        timestamp = datetime.utcnow().isoformat()
        for i, doc in enumerate(docs):
            doc.metadata = {
                **(doc.metadata or {}),
                "source_id": source_id,
                "source": source_name,
                "file_type": file_type,
                "ingested_at": timestamp,
                "doc_index": i,
            }
        return docs

    @staticmethod
    def load(file_path: str, file_type: str) -> List[Document]:
        if file_type == "pdf":
            docs = PyMuPDFLoader(file_path).load()
        elif file_type == "docx":
            docs = Docx2txtLoader(file_path).load()
        elif file_type == "txt":
            docs = TextLoader(file_path).load()
        elif file_type == "url":
            docs = WebBaseLoader(file_path).load()
        else:
            raise ValueError(f"Unsupported file type: {file_type}")

        if not docs:
            raise ValueError(
                f"Loader returned no documents for '{file_path}' (type={file_type})."
            )

        source_name = Path(file_path).name if file_type != "url" else file_path
        source_id = hashlib.sha1(file_path.encode()).hexdigest()[:12]
        return DocumentProcessor._enrich_metadata(docs, source_id, source_name, file_type)

    @staticmethod
    def extension_to_type(ext: str) -> Optional[str]:
        return {".pdf": "pdf", ".docx": "docx", ".txt": "txt"}.get(ext.lower())


# %%
# ============================================================================
# HYBRID INDEX — BM25 + Vector kept in sync
# ============================================================================
def _fallback_source_id(chunk: Document) -> str:
    content_key = chunk.page_content[:500]
    return "auto_" + hashlib.sha1(content_key.encode()).hexdigest()[:10]


class HybridIndex:
    """Single source of truth for BM25 + vector retrieval."""

    def __init__(self) -> None:
        self._lock = threading.RLock()
        self._chunks: List[Document] = []
        self._bm25: Optional[BM25Okapi] = None
        self._tokenized: List[List[str]] = []
        self._ingested_sources: set = set()

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return re.findall(r"\w+", text.lower())

    def _rebuild_bm25(self) -> None:
        if BM25_AVAILABLE and self._tokenized:
            self._bm25 = BM25Okapi(self._tokenized)

    def load_from_vector_store(self) -> None:
        """Re-sync BM25 index from persistent Chroma on startup."""
        try:
            with self._lock:
                collection = vector_store.get(include=["documents", "metadatas"])
                if not collection or not collection.get("documents"):
                    return
                docs = [
                    Document(page_content=text, metadata=meta or {})
                    for text, meta in zip(collection["documents"], collection["metadatas"])
                ]
                if not docs:
                    return
                self._chunks = docs
                self._tokenized = [self._tokenize(d.page_content) for d in docs]
                self._rebuild_bm25()
                self._ingested_sources = {
                    d.metadata.get("source_id", "")
                    for d in docs
                    if d.metadata.get("source_id")
                }
            logger.info(
                f"Reloaded {len(docs)} chunks from persistent vector store "
                f"({len(self._ingested_sources)} sources)."
            )
        except Exception as e:
            logger.warning(f"Could not reload from vector store: {e}")

    def add(self, chunks: List[Document]) -> int:
        """Deduplicate by source_id, then write to vector store and BM25."""
        if not chunks:
            return 0

        with self._lock:
            incoming_source_ids = {
                chunk.metadata.get("source_id") or _fallback_source_id(chunk)
                for chunk in chunks
            }
            already_ingested = incoming_source_ids & self._ingested_sources
            if already_ingested:
                logger.info(f"Skipping already-ingested sources: {already_ingested}")
                return 0

            new_chunks: List[Document] = []
            for chunk in chunks:
                sid = chunk.metadata.get("source_id") or _fallback_source_id(chunk)
                chunk.metadata["source_id"] = sid
                chunk.metadata["chunk_id"] = chunk.metadata.get("chunk_id") or str(uuid.uuid4())
                new_chunks.append(chunk)

            if not new_chunks:
                return 0

            try:
                vector_store.add_documents(new_chunks)
            except Exception as e:
                logger.error(f"Vector store write failed: {e}")
                raise

            self._chunks.extend(new_chunks)
            self._tokenized.extend(self._tokenize(c.page_content) for c in new_chunks)
            self._rebuild_bm25()
            for c in new_chunks:
                self._ingested_sources.add(c.metadata["source_id"])

            logger.info(
                f"Ingested {len(new_chunks)} chunks. "
                f"Total: {len(self._chunks)}. Sources: {len(self._ingested_sources)}"
            )
            return len(new_chunks)

    def vector_search(
        self,
        query: str,
        k: int,
        metadata_filter: Optional[Dict[str, Any]] = None,
    ) -> List[Tuple[Document, float]]:
        try:
            return vector_store.similarity_search_with_score(
                query, k=k, filter=metadata_filter or None
            )
        except Exception as e:
            logger.warning(f"Vector search failed: {e}")
            return []

    def bm25_search(
        self,
        query: str,
        k: int,
        metadata_filter: Optional[Dict[str, Any]] = None,
    ) -> List[Tuple[Document, float]]:
        if not (BM25_AVAILABLE and self._bm25 and self._chunks):
            return []
        with self._lock:
            tokens = self._tokenize(query)
            scores = self._bm25.get_scores(tokens)
            scored: List[Tuple[Document, float]] = []
            for chunk, score in zip(self._chunks, scores):
                if score <= 0:
                    continue
                if metadata_filter and not self._matches_filter(chunk, metadata_filter):
                    continue
                scored.append((chunk, float(score)))
            scored.sort(key=lambda x: x[1], reverse=True)
            return scored[:k]

    @staticmethod
    def _matches_filter(doc: Document, flt: Dict[str, Any]) -> bool:
        meta = doc.metadata or {}
        for key, val in flt.items():
            if isinstance(val, list):
                if meta.get(key) not in val:
                    return False
            elif meta.get(key) != val:
                return False
        return True

    @property
    def size(self) -> int:
        return len(self._chunks)

    @property
    def sources(self) -> List[str]:
        return sorted(self._ingested_sources)


hybrid_index = HybridIndex()
hybrid_index.load_from_vector_store()


# %%
# ============================================================================
# INGEST FUNCTIONS  ← THE MISSING PIECE (now defined before first use)
# ============================================================================
def ingest_documents(file_path: str, file_type: str) -> int:
    """
    Load a file or URL, chunk it, and add it to the hybrid index.
    Returns the number of new chunks added (0 if already indexed).
    """
    docs = DocumentProcessor.load(file_path, file_type)
    chunks = ChunkOptimizer.split(docs, file_type)
    return hybrid_index.add(chunks)


def ingest_file(file_path: str, file_type: str) -> Dict[str, Any]:
    """
    Convenience wrapper used by the interactive loop's 'ingest' command.
    Returns a stats dict.
    """
    n = ingest_documents(file_path, file_type)
    return {
        "chunks_added": n,
        "total_chunks": hybrid_index.size,
        "sources": hybrid_index.sources,
    }


# %%
# ============================================================================
# AUTO-INGESTION FROM DATA DIRECTORY
# ============================================================================
def auto_ingest_data_dir() -> Dict[str, Any]:
    """
    Scan CONFIG.data_dir for all supported files and ingest them.
    Already-ingested files are skipped (deduplication by source_id).
    Returns a summary dict.
    """
    summary: Dict[str, Any] = {"ingested": [], "skipped": [], "failed": []}
    data_path = CONFIG.data_dir

    if not data_path.exists():
        logger.warning(f"Data directory does not exist: {data_path}")
        return summary

    files = [
        f for f in data_path.iterdir()
        if f.is_file() and f.suffix.lower() in CONFIG.supported_extensions
    ]

    if not files:
        logger.info(f"No supported files found in {data_path}")
        return summary

    logger.info(f"Found {len(files)} file(s) in data directory. Ingesting...")

    for file_path in files:
        file_type = DocumentProcessor.extension_to_type(file_path.suffix)
        if not file_type:
            summary["skipped"].append(str(file_path))
            continue
        try:
            n = ingest_documents(str(file_path), file_type)
            if n > 0:
                logger.info(f"  ✓ Ingested '{file_path.name}' → {n} chunks")
                summary["ingested"].append(file_path.name)
            else:
                logger.info(f"  — Skipped '{file_path.name}' (already indexed)")
                summary["skipped"].append(file_path.name)
        except Exception as e:
            logger.error(f"  ✗ Failed '{file_path.name}': {e}")
            summary["failed"].append({"file": file_path.name, "error": str(e)})

    logger.info(
        f"Auto-ingestion complete. "
        f"New: {len(summary['ingested'])}, "
        f"Skipped: {len(summary['skipped'])}, "
        f"Failed: {len(summary['failed'])}. "
        f"Total chunks: {hybrid_index.size}"
    )
    return summary


# %%
# ============================================================================
# RETRIEVAL UTILITIES
# ============================================================================
def reciprocal_rank_fusion(
    vec_results: List[Document],
    bm25_results: List[Document],
    k: int = CONFIG.rrf_k,
    bm25_weight: float = CONFIG.bm25_weight,
) -> List[Document]:
    """
    Weighted RRF fusion. For each document:
        score += weight * (1 / (k + rank + 1))
    bm25_weight applies to BM25; (1 - bm25_weight) applies to vector.
    """
    vec_weight = max(0.0, 1.0 - bm25_weight)
    bm25_w = max(0.0, bm25_weight)

    fused: Dict[str, Tuple[Document, float]] = {}

    for rank, doc in enumerate(vec_results):
        doc_id = doc.metadata.get("chunk_id") or doc.page_content[:100]
        score = vec_weight * (1.0 / (k + rank + 1))
        if doc_id in fused:
            fused[doc_id] = (fused[doc_id][0], fused[doc_id][1] + score)
        else:
            fused[doc_id] = (doc, score)

    for rank, doc in enumerate(bm25_results):
        doc_id = doc.metadata.get("chunk_id") or doc.page_content[:100]
        score = bm25_w * (1.0 / (k + rank + 1))
        if doc_id in fused:
            fused[doc_id] = (fused[doc_id][0], fused[doc_id][1] + score)
        else:
            fused[doc_id] = (doc, score)

    return [doc for doc, _ in sorted(fused.values(), key=lambda x: x[1], reverse=True)]


def hybrid_retrieve(
    query: str,
    k: int,
    metadata_filter: Optional[Dict[str, Any]] = None,
) -> List[Document]:
    """Run BM25 + vector search and fuse with weighted RRF."""
    vec = [doc for doc, _ in hybrid_index.vector_search(query, k, metadata_filter)]
    bm = [doc for doc, _ in hybrid_index.bm25_search(query, k, metadata_filter)]
    return reciprocal_rank_fusion(vec, bm)


def rerank_documents(query: str, docs: List[Document], top_k: int) -> List[Document]:
    if not docs:
        return []
    if reranker is None or len(docs) <= 1:
        return docs[:top_k]
    try:
        pairs = [(query, d.page_content) for d in docs]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
        return [d for _, d in ranked[:top_k]]
    except Exception as e:
        logger.warning(f"Reranking failed: {e}")
        return docs[:top_k]


def format_source_label(meta: Dict[str, Any]) -> str:
    """Build a human-readable source label from metadata."""
    src = meta.get("source", "unknown")
    page = meta.get("page")
    if page is not None:
        return f"{src} (p. {int(page) + 1})"
    return src


def format_context_with_sources(docs: List[Document]) -> Tuple[str, List[Dict[str, Any]]]:
    """Build a context string with citation markers [1], [2], ..."""
    if not docs:
        return "", []

    blocks: List[str] = []
    sources: List[Dict[str, Any]] = []
    for i, doc in enumerate(docs, start=1):
        label = format_source_label(doc.metadata or {})
        blocks.append(f"[{i}] Source: {label}\n{doc.page_content.strip()}")
        sources.append(
            {
                "id": i,
                "label": label,
                "source": (doc.metadata or {}).get("source"),
                "page": (doc.metadata or {}).get("page"),
                "chunk_id": (doc.metadata or {}).get("chunk_id"),
                "preview": doc.page_content.strip()[:200],
            }
        )
    return "\n\n---\n\n".join(blocks), sources


# %%
# ============================================================================
# AGENT STATE DEFINITIONS
# ============================================================================
class BaseState(TypedDict, total=False):
    input: str
    file_path: Optional[str]
    file_type: Optional[Literal["pdf", "docx", "txt", "url"]]
    history: List[Dict[str, str]]
    answer: Optional[str]
    sources: Optional[List[Dict[str, Any]]]
    current_agent: Optional[str]
    next_agent: Optional[str]


class NaiveState(BaseState, total=False):
    context_docs: Optional[List[Document]]
    metadata_filter: Optional[Dict[str, Any]]


class AdvancedState(BaseState, total=False):
    rewritten_queries: Optional[List[str]]
    context_docs: Optional[List[Document]]
    metadata_filter: Optional[Dict[str, Any]]


class CRAGState(BaseState, total=False):
    context_docs: Optional[List[Document]]
    retrieval_score: Optional[float]
    retry_count: int
    metadata_filter: Optional[Dict[str, Any]]


class SelfRAGState(BaseState, total=False):
    need_retrieval: Optional[bool]
    context_docs: Optional[List[Document]]
    draft_answer: Optional[str]
    critique: Optional[str]
    confidence: Optional[float]
    metadata_filter: Optional[Dict[str, Any]]


# %%
# ============================================================================
# CONVERSATIONAL MEMORY HELPER
# ============================================================================
def _history_messages(history: List[Dict[str, str]]) -> list:
    """Convert history list into LangChain message tuples."""
    return [
        ("human" if h["role"] == "user" else "assistant", h["content"])
        for h in history
    ]


def finalize(state: BaseState) -> BaseState:
    """Append the current turn to conversational history."""
    history = list(state.get("history") or [])
    history.append({"role": "user", "content": state["input"]})
    history.append({"role": "assistant", "content": state.get("answer") or ""})
    return {**state, "history": history}


# %%
# ============================================================================
# 1. NAIVE RAG
# ============================================================================
def naive_router(state: NaiveState) -> NaiveState:
    fp, ft = state.get("file_path"), state.get("file_type")
    if bool(fp) ^ bool(ft):
        logger.warning("naive_router: file_path and file_type must both be set. Ignoring file.")
    if fp and ft:
        return {**state, "current_agent": "ingest"}
    if hybrid_index.size > 0:
        return {**state, "current_agent": "retrieve"}
    return {**state, "current_agent": "fallback"}


def naive_ingest(state: NaiveState) -> NaiveState:
    try:
        ingest_documents(state["file_path"], state["file_type"])
    except Exception as e:
        logger.error(f"Ingestion failed: {e}")
    if hybrid_index.size > 0:
        return {**state, "current_agent": "retrieve"}
    return {**state, "current_agent": "fallback"}


def naive_retrieve(state: NaiveState) -> NaiveState:
    """Full hybrid retrieval: BM25 + Vector → weighted RRF → rerank."""
    query = state["input"]
    fused = hybrid_retrieve(query, CONFIG.initial_k, state.get("metadata_filter"))
    docs = rerank_documents(query, fused, CONFIG.rerank_top_k)
    return {**state, "context_docs": docs, "current_agent": "answer"}


def naive_answer(state: NaiveState) -> NaiveState:
    docs = state.get("context_docs") or []
    context_text, sources = format_context_with_sources(docs)
    history = state.get("history") or []

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a helpful assistant. Answer using only the provided context. "
         "Cite sources inline using bracketed numbers e.g. [1], [2]. "
         "If the context is insufficient, say so clearly."),
        *_history_messages(history),
        ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer (cite sources):"),
    ])
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context_text or "No context available.", "question": state["input"]})
    return {**state, "answer": answer, "sources": sources, "current_agent": "finalize"}


def naive_fallback(state: NaiveState) -> NaiveState:
    try:
        wiki = wikipedia.run(state["input"])
        answer = f"No documents indexed. From Wikipedia:\n\n{wiki}"
        sources = [{"id": 1, "label": "Wikipedia", "source": "wikipedia"}]
    except Exception:
        answer = "I couldn't find any information to answer your question."
        sources = []
    return {**state, "answer": answer, "sources": sources, "current_agent": "finalize"}


def build_naive_rag():
    g = StateGraph(NaiveState)
    g.add_node("router", naive_router)
    g.add_node("ingest", naive_ingest)
    g.add_node("retrieve", naive_retrieve)
    g.add_node("answer", naive_answer)
    g.add_node("fallback", naive_fallback)
    g.add_node("finalize", finalize)
    g.set_entry_point("router")
    g.add_conditional_edges("router", lambda s: s["current_agent"],
                            {"ingest": "ingest", "retrieve": "retrieve", "fallback": "fallback"})
    g.add_conditional_edges("ingest", lambda s: s["current_agent"],
                            {"retrieve": "retrieve", "fallback": "fallback"})
    g.add_edge("retrieve", "answer")
    g.add_edge("answer", "finalize")
    g.add_edge("fallback", "finalize")
    g.add_edge("finalize", END)
    return g.compile()


naive_rag_app = build_naive_rag()


# %%
# ============================================================================
# 2. ADVANCED RAG
# ============================================================================
def advanced_ingest(state: AdvancedState) -> AdvancedState:
    if state.get("file_path") and state.get("file_type"):
        try:
            ingest_documents(state["file_path"], state["file_type"])
        except Exception as e:
            logger.error(f"Advanced ingestion failed: {e}")
    return {**state, "current_agent": "rewrite"}


def advanced_rewrite(state: AdvancedState) -> AdvancedState:
    """Multi-query expansion for better recall."""
    original = state["input"]
    history = state.get("history") or []

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a query optimization expert. Given the conversation history and the "
         "current question, generate exactly 3 diverse reformulations for retrieval. "
         "Each on its own line, no numbering, no preamble. "
         "Vary phrasing, expand abbreviations, make implicit intent explicit."),
        *_history_messages(history[-4:]),
        ("human", "{query}"),
    ])
    chain = prompt | llm | StrOutputParser()
    try:
        raw = chain.invoke({"query": original})
        variants = [line.strip() for line in raw.splitlines() if line.strip()][:3]
    except Exception as e:
        logger.warning(f"Query rewrite failed: {e}")
        variants = [original]

    queries = [original] + [v for v in variants if v.lower() != original.lower()]
    return {**state, "rewritten_queries": queries, "current_agent": "retrieve"}


def advanced_retrieve(state: AdvancedState) -> AdvancedState:
    """Per-query RRF then cross-query merge."""
    queries = state.get("rewritten_queries") or [state["input"]]
    metadata_filter = state.get("metadata_filter")
    per_query_fused: List[List[Document]] = []

    for q in queries:
        vec = [doc for doc, _ in hybrid_index.vector_search(q, CONFIG.initial_k, metadata_filter)]
        bm = [doc for doc, _ in hybrid_index.bm25_search(q, CONFIG.initial_k, metadata_filter)]
        if vec or bm:
            per_query_fused.append(reciprocal_rank_fusion(vec, bm))

    if not per_query_fused:
        return {**state, "context_docs": [], "current_agent": "fallback"}

    merged: List[Document] = per_query_fused[0]
    for nxt in per_query_fused[1:]:
        merged = reciprocal_rank_fusion(merged, nxt)

    seen: set = set()
    deduped: List[Document] = []
    for doc in merged:
        did = doc.metadata.get("chunk_id") or doc.page_content[:100]
        if did not in seen:
            deduped.append(doc)
            seen.add(did)

    reranked = rerank_documents(state["input"], deduped, CONFIG.rerank_top_k)
    nxt = "compress" if reranked else "fallback"
    return {**state, "context_docs": reranked, "current_agent": nxt}


def _compress_one(args: Tuple[str, Document, float]) -> Optional[Document]:
    query, doc, delay = args
    if delay > 0:
        time.sleep(delay)
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Extract only the sentences directly relevant to the query. "
         "Preserve original wording. If nothing is relevant, return NONE."),
        ("human", "Query: {query}\n\nPassage:\n{passage}"),
    ])
    chain = prompt | llm | StrOutputParser()
    try:
        out = chain.invoke({"query": query, "passage": doc.page_content}).strip()
        if not out or out.upper().startswith("NONE"):
            return None
        return Document(page_content=out, metadata=dict(doc.metadata or {}))
    except Exception as e:
        logger.warning(f"Compression failed for one chunk: {e}")
        return doc


def advanced_compress(state: AdvancedState) -> AdvancedState:
    docs = state.get("context_docs") or []
    if not docs or not CONFIG.compression_enabled:
        return {**state, "current_agent": "answer"}
    query = state["input"]
    delay = CONFIG.compress_request_delay_s
    args = [(query, d, i * delay) for i, d in enumerate(docs)]
    workers = max(1, CONFIG.compression_max_workers)
    if workers == 1:
        results = [_compress_one(a) for a in args]
    else:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            results = list(ex.map(_compress_one, args))
    compressed = [r for r in results if r is not None]
    return {**state, "context_docs": compressed or docs, "current_agent": "answer"}


def advanced_answer(state: AdvancedState) -> AdvancedState:
    docs = state.get("context_docs") or []
    context_text, sources = format_context_with_sources(docs)
    history = state.get("history") or []

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are an expert assistant with curated, reranked, compressed context. "
         "Answer accurately and cite sources inline using bracketed numbers [1], [2]. "
         "If the context is insufficient, say so explicitly."),
        *_history_messages(history),
        ("human", "Context:\n{context}\n\nQuestion: {question}\n\nComprehensive answer with inline citations:"),
    ])
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context_text or "No relevant context.", "question": state["input"]})
    return {**state, "answer": answer, "sources": sources, "current_agent": "finalize"}


def advanced_fallback(state: AdvancedState) -> AdvancedState:
    try:
        wiki = wikipedia.run(state["input"])
        answer = f"No relevant documents indexed. From Wikipedia:\n\n{wiki}"
        sources = [{"id": 1, "label": "Wikipedia", "source": "wikipedia"}]
    except Exception:
        answer = "I couldn't find any relevant information."
        sources = []
    return {**state, "answer": answer, "sources": sources, "current_agent": "finalize"}


def build_advanced_rag():
    g = StateGraph(AdvancedState)
    g.add_node("ingest", advanced_ingest)
    g.add_node("rewrite", advanced_rewrite)
    g.add_node("retrieve", advanced_retrieve)
    g.add_node("compress", advanced_compress)
    g.add_node("answer", advanced_answer)
    g.add_node("fallback", advanced_fallback)
    g.add_node("finalize", finalize)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "rewrite")
    g.add_edge("rewrite", "retrieve")
    g.add_conditional_edges("retrieve", lambda s: s["current_agent"],
                            {"compress": "compress", "fallback": "fallback"})
    g.add_edge("compress", "answer")
    g.add_edge("answer", "finalize")
    g.add_edge("fallback", "finalize")
    g.add_edge("finalize", END)
    return g.compile()


advanced_rag_app = build_advanced_rag()


# %%
# ============================================================================
# 3. CORRECTIVE RAG (CRAG)
# ============================================================================
def crag_ingest(state: CRAGState) -> CRAGState:
    if state.get("file_path") and state.get("file_type"):
        try:
            ingest_documents(state["file_path"], state["file_type"])
        except Exception as e:
            logger.error(f"CRAG ingestion failed: {e}")
    return state


def crag_retrieve(state: CRAGState) -> CRAGState:
    if hybrid_index.size == 0:
        return {**state, "context_docs": [], "retrieval_score": 0.0}
    metadata_filter = state.get("metadata_filter")
    fused = hybrid_retrieve(state["input"], CONFIG.initial_k, metadata_filter)
    reranked = rerank_documents(state["input"], fused, CONFIG.rerank_top_k)
    raw = hybrid_index.vector_search(state["input"], k=5, metadata_filter=metadata_filter)
    if raw:
        avg_score = sum(s for _, s in raw) / len(raw)
        vector_confidence = max(0.0, 1.0 - avg_score)
    else:
        vector_confidence = 0.0
    return {**state, "context_docs": reranked, "retrieval_score": vector_confidence}


def crag_evaluate(state: CRAGState) -> CRAGState:
    docs = state.get("context_docs") or []
    vec_conf = state.get("retrieval_score", 0.0)
    history = state.get("history") or []

    if not docs:
        return {**state, "retrieval_score": 0.0, "next_agent": "web_search"}

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Rate how relevant the retrieved context is for answering the question, "
         "on a scale 0.0 to 1.0. Respond with ONLY a decimal number."),
        *_history_messages(history[-2:]),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ])
    chain = prompt | llm | StrOutputParser()
    try:
        ctx = "\n\n".join(d.page_content for d in docs[:3])
        raw = chain.invoke({"question": state["input"], "context": ctx})
        match = re.search(r"[0-9]*\.?[0-9]+", raw)
        llm_score = max(0.0, min(1.0, float(match.group(0)))) if match else 0.5
    except Exception as e:
        logger.warning(f"CRAG evaluator LLM failed: {e}")
        llm_score = 0.5

    final = 0.4 * vec_conf + 0.6 * llm_score
    retries = state.get("retry_count", 0)

    if final >= CONFIG.crag_high_confidence:
        nxt = "answer"
    elif retries >= CONFIG.crag_max_retries:
        logger.warning(f"Max retries reached. Proceeding with confidence {final:.2f}")
        nxt = "answer"
    else:
        logger.info(f"Low confidence ({final:.2f}), web search. Retry {retries+1}/{CONFIG.crag_max_retries}")
        nxt = "web_search"

    return {**state, "retrieval_score": final, "next_agent": nxt}


def crag_web_search(state: CRAGState) -> CRAGState:
    question = state["input"]
    retries = state.get("retry_count", 0)

    if retries > 0:
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Rewrite this question as a more specific web search query. Return ONLY the rewritten query."),
            ("human", "{question}"),
        ])
        try:
            question = (prompt | llm | StrOutputParser()).invoke({"question": question}).strip() or question
        except Exception:
            pass

    web_doc: Optional[Document] = None
    wiki_content = ""
    try:
        wiki_content = wikipedia.run(question)
        web_doc = Document(
            page_content=wiki_content,
            metadata={"source": "Wikipedia", "source_id": "wikipedia", "file_type": "web"},
        )
    except Exception as e:
        logger.warning(f"Wikipedia lookup failed: {e}")

    existing = state.get("context_docs") or []
    combined = existing + ([web_doc] if web_doc else [])

    current_score = state.get("retrieval_score", 0.0)
    if len(wiki_content) > 500:
        boost = min(0.2, len(wiki_content) / 10000)
        new_score = min(1.0, current_score + boost)
    elif len(wiki_content) > 200:
        new_score = min(1.0, current_score + 0.05)
    else:
        new_score = max(0.0, current_score - 0.1)
        logger.warning("Wikipedia returned minimal content — score penalized.")

    return {**state, "context_docs": combined, "retry_count": retries + 1, "retrieval_score": new_score}


def crag_answer(state: CRAGState) -> CRAGState:
    docs = state.get("context_docs") or []
    score = state.get("retrieval_score", 0.0)
    history = state.get("history") or []

    if score >= CONFIG.crag_high_confidence:
        confidence_note = "The context is highly relevant — answer confidently."
    elif score >= CONFIG.crag_low_confidence:
        confidence_note = "The context is partially relevant — flag any uncertainty."
    else:
        confidence_note = "Context quality is low — be explicit about uncertainty."

    context_text, sources = format_context_with_sources(docs)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         f"You are a careful, accurate assistant. {confidence_note} "
         "Always cite sources inline using bracketed numbers [1], [2]."),
        *_history_messages(history),
        ("human", "Context (Confidence: {score_pct}):\n{context}\n\nQuestion: {question}\n\nAnswer with citations:"),
    ])
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "context": context_text or "No context available.",
        "question": state["input"],
        "score_pct": f"{score:.0%}",
    })
    return {**state, "answer": answer, "sources": sources}


def build_crag():
    g = StateGraph(CRAGState)
    g.add_node("ingest", crag_ingest)
    g.add_node("retrieve", crag_retrieve)
    g.add_node("evaluate", crag_evaluate)
    g.add_node("web_search", crag_web_search)
    g.add_node("answer", crag_answer)
    g.add_node("finalize", finalize)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "retrieve")
    g.add_edge("retrieve", "evaluate")
    g.add_conditional_edges("evaluate", lambda s: s["next_agent"],
                            {"answer": "answer", "web_search": "web_search"})
    g.add_edge("web_search", "evaluate")
    g.add_edge("answer", "finalize")
    g.add_edge("finalize", END)
    return g.compile()


crag_app = build_crag()


# %%
# ============================================================================
# 4. SELF-RAG
# ============================================================================
def self_rag_ingest(state: SelfRAGState) -> SelfRAGState:
    if state.get("file_path") and state.get("file_type"):
        try:
            ingest_documents(state["file_path"], state["file_type"])
        except Exception as e:
            logger.error(f"Self-RAG ingestion failed: {e}")
    return state


def self_rag_decide(state: SelfRAGState) -> SelfRAGState:
    """Decide whether retrieval is needed, considering conversation history."""
    has_docs = hybrid_index.size > 0
    history = state.get("history") or []
    history_text = ""
    if history:
        recent = history[-6:]
        history_text = "\n".join(f"{h['role'].capitalize()}: {h['content']}" for h in recent)
        history_text = f"\nConversation so far:\n{history_text}\n"

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Decide if external document retrieval is needed to answer the question. "
         "Answer YES for: questions about uploaded documents, specific facts, technical details, "
         "or follow-up questions on topics that needed retrieval previously. "
         "Answer NO for: simple general knowledge, conversational messages, or questions "
         "you can answer confidently without external documents. "
         "Respond with ONLY 'YES' or 'NO'."),
        ("human", "{history_and_question}"),
    ])
    chain = prompt | llm | StrOutputParser()
    try:
        combined = f"{history_text}Current question: {state['input']}"
        decision = chain.invoke({"history_and_question": combined}).strip().upper()
        need = "YES" in decision and has_docs
    except Exception:
        need = has_docs
    return {**state, "need_retrieval": need}


def self_rag_retrieve(state: SelfRAGState) -> SelfRAGState:
    metadata_filter = state.get("metadata_filter")
    fused = hybrid_retrieve(state["input"], CONFIG.initial_k, metadata_filter)
    reranked = rerank_documents(state["input"], fused, CONFIG.rerank_top_k)
    return {**state, "context_docs": reranked}


def self_rag_generate(state: SelfRAGState) -> SelfRAGState:
    docs = state.get("context_docs") or []
    history = state.get("history") or []
    context_text, sources = format_context_with_sources(docs)

    if docs:
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Generate a comprehensive answer based on the context. Cite sources inline [1], [2]."),
            *_history_messages(history),
            ("human", "Context:\n{context}\n\nQuestion: {question}\n\nDraft answer:"),
        ])
        chain = prompt | llm | StrOutputParser()
        draft = chain.invoke({"context": context_text, "question": state["input"]})
    else:
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Answer the question from your own knowledge."),
            *_history_messages(history),
            ("human", "Question: {question}\n\nAnswer:"),
        ])
        chain = prompt | llm | StrOutputParser()
        draft = chain.invoke({"question": state["input"]})

    return {**state, "draft_answer": draft, "sources": sources}


def self_rag_critique(state: SelfRAGState) -> SelfRAGState:
    draft = state.get("draft_answer", "")
    docs = state.get("context_docs") or []
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Critique the draft answer for accuracy, completeness, hallucinations, and clarity. "
         "Use this exact format:\n"
         "CRITIQUE: <your critique>\n"
         "CONFIDENCE: <0.0 to 1.0>\n"
         "NEEDS_REVISION: <YES or NO>"),
        ("human", "Question: {question}\n\nContext:\n{context}\n\nDraft:\n{draft}\n\nEvaluate:"),
    ])
    chain = prompt | llm | StrOutputParser()
    try:
        ctx = "\n".join(d.page_content for d in docs[:3]) if docs else "(none)"
        out = chain.invoke({"question": state["input"], "context": ctx, "draft": draft})
        m = re.search(r"CONFIDENCE:\s*([0-9]*\.?[0-9]+)", out)
        confidence = max(0.0, min(1.0, float(m.group(1)))) if m else 0.7
        needs = bool(re.search(r"NEEDS_REVISION:\s*YES", out, re.IGNORECASE))
    except Exception as e:
        logger.warning(f"Self-RAG critique failed: {e}")
        out = "Unable to generate critique."
        confidence = 0.7
        needs = False

    return {**state, "critique": out, "confidence": confidence, "answer": draft,
            "next_agent": "revise" if needs else "finalize"}


def self_rag_revise(state: SelfRAGState) -> SelfRAGState:
    docs = state.get("context_docs") or []
    context_text, _ = format_context_with_sources(docs)
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Revise the draft based on the critique. Address every critique point, "
         "ground claims in the context, avoid unsupported assertions, "
         "keep inline citations [1], [2]."),
        ("human", "Question: {question}\n\nContext:\n{context}\n\nDraft:\n{draft}\n\nCritique:\n{critique}\n\nRevised answer:"),
    ])
    chain = prompt | llm | StrOutputParser()
    try:
        revised = chain.invoke({
            "question": state["input"],
            "context": context_text or "(none)",
            "draft": state.get("draft_answer", ""),
            "critique": state.get("critique", ""),
        })
    except Exception as e:
        logger.warning(f"Self-RAG revision failed: {e}")
        revised = state.get("draft_answer", "")
    return {**state, "answer": revised, "draft_answer": revised}


def build_self_rag():
    g = StateGraph(SelfRAGState)
    g.add_node("ingest", self_rag_ingest)
    g.add_node("decide", self_rag_decide)
    g.add_node("retrieve", self_rag_retrieve)
    g.add_node("generate", self_rag_generate)
    g.add_node("critique", self_rag_critique)
    g.add_node("revise", self_rag_revise)
    g.add_node("finalize", finalize)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "decide")
    g.add_conditional_edges("decide",
                            lambda s: "retrieve" if s.get("need_retrieval") else "generate",
                            {"retrieve": "retrieve", "generate": "generate"})
    g.add_edge("retrieve", "generate")
    g.add_edge("generate", "critique")
    g.add_conditional_edges("critique", lambda s: s["next_agent"],
                            {"revise": "revise", "finalize": "finalize"})
    g.add_edge("revise", "finalize")
    g.add_edge("finalize", END)
    return g.compile()


self_rag_app = build_self_rag()


# %%
# ============================================================================
# UNIFIED INTERFACE
# ============================================================================
def process_query(
    input_text: str,
    file_path: Optional[str] = None,
    file_type: Optional[str] = None,
    mode: Literal["naive", "advanced", "crag", "self_rag"] = "advanced",
    history: Optional[List[Dict[str, str]]] = None,
    metadata_filter: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Unified entry point for all RAG modes.

    Args:
        input_text:       The user's question.
        file_path:        Path to a document or URL (optional — data/ folder is auto-ingested).
        file_type:        One of 'pdf', 'docx', 'txt', 'url'.
        mode:             'naive' | 'advanced' | 'crag' | 'self_rag'.
        history:          Prior turns [{"role": "user"|"assistant", "content": "..."}].
                          Pass the returned history back on every call for memory.
        metadata_filter:  Restrict retrieval e.g. {"file_type": "pdf"}.

    Returns:
        Dict with: answer, sources, history, mode, indexed_sources, metadata.
    """
    history = history or []
    common: Dict[str, Any] = {
        "input": input_text,
        "file_path": file_path,
        "file_type": file_type,
        "history": history,
        "metadata_filter": metadata_filter,
    }

    if mode == "naive":
        result = naive_rag_app.invoke(common)
        meta: Dict[str, Any] = {}
    elif mode == "advanced":
        result = advanced_rag_app.invoke(common)
        meta = {"rewritten_queries": result.get("rewritten_queries")}
    elif mode == "crag":
        result = crag_app.invoke({**common, "retry_count": 0})
        meta = {"retrieval_score": result.get("retrieval_score"), "retry_count": result.get("retry_count")}
    elif mode == "self_rag":
        result = self_rag_app.invoke(common)
        meta = {"need_retrieval": result.get("need_retrieval"), "confidence": result.get("confidence"),
                "critique": result.get("critique")}
    else:
        raise ValueError(f"Unknown mode: {mode!r}")

    return {
        "answer": result.get("answer", "No answer generated."),
        "sources": result.get("sources", []),
        "history": result.get("history", history),
        "mode": mode,
        "indexed_sources": hybrid_index.sources,
        "metadata": meta,
    }


# %%
# ============================================================================
# PRINT HELPER
# ============================================================================
def _print_result(res: Dict[str, Any]) -> None:
    print(f"\n{'─' * 70}")
    print(f"[Mode: {res['mode']}]")
    print(f"{'─' * 70}")
    print(f"Answer:\n{res['answer']}\n")
    if res.get("sources"):
        print("Sources:")
        for s in res["sources"]:
            print(f"  [{s['id']}] {s.get('label', s.get('source', 'unknown'))}")
    if res.get("metadata"):
        filtered_meta = {k: v for k, v in res["metadata"].items() if v is not None and k != "critique"}
        if filtered_meta:
            print(f"Metadata: {filtered_meta}")
    if res.get("indexed_sources"):
        print(f"Indexed sources: {res['indexed_sources']}")
    print()


# ============================================================================
# INTERACTIVE LOOP
# ============================================================================
def _print_help() -> None:
    print("""
Commands:
  ask <question>                          Ask using the current mode
  mode <naive|advanced|crag|self_rag>     Switch RAG mode
  ingest <path-or-url> <type>             Ingest a file manually (type: pdf|docx|txt|url)
  filter <key=value | clear>              Set / clear metadata filter
  history                                 Show conversation history
  reset                                   Clear conversation history
  status                                  Show current settings and index info
  help                                    Show this help message
  quit / exit                             Exit
""")


def interactive_loop() -> None:
    print("=" * 70)
    print("  ADVANCED RAG SYSTEM — Interactive Mode")
    print(f"  Data directory   : {CONFIG.data_dir}")
    print(f"  Vector store     : {CONFIG.vector_store_dir}")
    print("  Modes            : naive | advanced | crag | self_rag")
    print("  Type 'help' for commands, 'quit' to exit")
    print("=" * 70)

    print("\nScanning data directory for documents...")
    auto_ingest_data_dir()
    print(f"Total chunks indexed: {hybrid_index.size}")
    print(f"Indexed sources     : {hybrid_index.sources or '(none)'}\n")

    mode: str = "advanced"
    history: List[Dict[str, str]] = []
    metadata_filter: Optional[Dict[str, Any]] = None

    print(f"Current mode: {mode}")
    print("Tip: Just type your question directly, or use 'ask <question>'.\n")

    while True:
        try:
            raw = input(f"[{mode}]> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not raw:
            continue

        parts = raw.split(maxsplit=1)
        cmd = parts[0].lower()
        rest = parts[1] if len(parts) > 1 else ""

        if cmd in ("quit", "exit"):
            print("Goodbye!")
            break

        elif cmd == "help":
            _print_help()

        elif cmd == "status":
            print(f"  Mode             : {mode}")
            print(f"  Metadata filter  : {metadata_filter or '(none)'}")
            print(f"  History turns    : {len(history) // 2}")
            print(f"  Indexed sources  : {hybrid_index.sources or '(none)'}")
            print(f"  Total chunks     : {hybrid_index.size}")
            print(f"  Data directory   : {CONFIG.data_dir}\n")

        elif cmd == "reset":
            history = []
            print("Conversation history cleared.\n")

        elif cmd == "history":
            if not history:
                print("(empty)\n")
            else:
                for h in history:
                    role = h["role"].upper()
                    content = h["content"][:300]
                    print(f"  {role}: {content}")
                print()

        elif cmd == "mode":
            new_mode = rest.strip().lower()
            if new_mode in ("naive", "advanced", "crag", "self_rag"):
                mode = new_mode
                print(f"Switched to mode: {mode}\n")
            else:
                print("Invalid mode. Choose: naive | advanced | crag | self_rag\n")

        elif cmd == "ingest":
            ing_parts = rest.split()
            if len(ing_parts) != 2:
                print("Usage: ingest <path-or-url> <pdf|docx|txt|url>\n")
                continue
            path, ftype = ing_parts
            if ftype not in ("pdf", "docx", "txt", "url"):
                print("Invalid file type. Use: pdf | docx | txt | url\n")
                continue
            try:
                stats = ingest_file(path, ftype)
                print(f"Ingested. Stats: {stats}\n")
            except Exception as e:
                print(f"Ingestion failed: {e}\n")

        elif cmd == "filter":
            arg = rest.strip()
            if arg.lower() == "clear" or not arg:
                metadata_filter = None
                print("Filter cleared.\n")
            elif "=" in arg:
                key, val = arg.split("=", 1)
                metadata_filter = {key.strip(): val.strip()}
                print(f"Filter set: {metadata_filter}\n")
            else:
                print("Usage: filter <key=value> | filter clear\n")

        elif cmd == "ask":
            question = rest.strip()
            if not question:
                print("Usage: ask <your question>\n")
                continue
            try:
                res = process_query(
                    input_text=question,
                    mode=mode,
                    history=history,
                    metadata_filter=metadata_filter,
                )
                _print_result(res)
                history = res["history"]
            except Exception as e:
                print(f"Query failed: {e}\n")

        else:
            # Treat any unrecognized input as a direct question
            try:
                res = process_query(
                    input_text=raw,
                    mode=mode,
                    history=history,
                    metadata_filter=metadata_filter,
                )
                _print_result(res)
                history = res["history"]
            except Exception as e:
                print(f"Query failed: {e}\n")


# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    interactive_loop()

2026-05-07 12:58:43,994 | WARNING | langchain_community.utils.user_agent | USER_AGENT environment variable not set, consider setting it to identify your requests.
2026-05-07 12:58:44,481 | INFO    | rag | Data directory   : c:\Users\codix\OneDrive\Desktop\New folder (2)\AutoDocThinker\backend\data
2026-05-07 12:58:44,482 | INFO    | rag | Vector store dir : c:\Users\codix\OneDrive\Desktop\New folder (2)\AutoDocThinker\backend\data\vector_store
2026-05-07 12:58:44,486 | INFO    | sentence_transformers.SentenceTransformer | Use pytorch device_name: cpu
2026-05-07 12:58:44,486 | INFO    | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-05-07 12:58:50,909 | INFO    | chromadb.telemetry.product.posthog | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-05-07 12:58:54,130 | INFO    | sentence_transformers.cross_encoder.CrossEncoder | Use pytorch device: cpu
2026-05-07 1

  ADVANCED RAG SYSTEM — Interactive Mode
  Data directory   : c:\Users\codix\OneDrive\Desktop\New folder (2)\AutoDocThinker\backend\data
  Vector store     : c:\Users\codix\OneDrive\Desktop\New folder (2)\AutoDocThinker\backend\data\vector_store
  Modes            : naive | advanced | crag | self_rag
  Type 'help' for commands, 'quit' to exit

Scanning data directory for documents...


C:\Users\codix\AppData\Local\Temp\ipykernel_16540\2916559477.py:250: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat()
2026-05-07 12:58:55,557 | INFO    | rag | Ingested 6 chunks. Total: 6. Sources: 1
2026-05-07 12:58:55,558 | INFO    | rag |   ✓ Ingested 'Md Hasan Imon - (AI & ML) Resume.pdf' → 6 chunks
2026-05-07 12:58:55,558 | INFO    | rag | Auto-ingestion complete. New: 1, Skipped: 0, Failed: 0. Total chunks: 6


Total chunks indexed: 6
Indexed sources     : ['8c6ac47ea33c']

Current mode: advanced
Tip: Just type your question directly, or use 'ask <question>'.



2026-05-07 12:59:42,262 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-07 12:59:43,186 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-07 12:59:43,513 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-07 12:59:43,899 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-07 12:59:43,904 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-07 12:59:43,917 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-07 12:59:45,003 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



──────────────────────────────────────────────────────────────────────
[Mode: advanced]
──────────────────────────────────────────────────────────────────────
Answer:
Md Hasan Imon is an aspiring AI/ML engineer who has already built a solid portfolio of hands‑on projects in generative and agentic artificial intelligence as well as classic machine‑learning pipelines.  

**Technical expertise**  
- **Generative & Agentic AI:** He has experience constructing production‑ready multi‑agent systems, retrieval‑augmented generation (RAG) pipelines, and memory‑augmented AI solutions that can perform advanced reasoning, planning, and decision‑making.  
- **Large‑language‑model fine‑tuning:** He works with parameter‑efficient fine‑tuning (PEFT) techniques such as LoRA and QLoRA to adapt large language models for specific tasks.  
- **Natural‑language processing:** His background includes text classification, sentiment analysis, transformer architectures, and language‑model training, enabling him 